# Topic: Secrets Management - Credentials hardcoding, environment variable loading, and secure config files
 
## 1. THE HARDCODED SECRETS RISK
- **The Vulnerability**: Hardcoding API keys, passwords, or private keys inside source files is one of the most common causes of enterprise database breaches.
  - Committing files to source control repositories (like GitHub) leaks secrets instantly. Automated crawlers scan public repositories for leaked keys.
 
## 2. ENVIRONMENT INJECTION
- **Best Practice**: Inject configurations dynamically at runtime using **Environment Variables**:
  - Avoid committing secret files.
  - Use a `.env` file for local development, and add it explicitly to `.gitignore`.
  - Parse the `.env` file or read environment variables directly using Python's standard `os.environ.get()`.
 
---


In [ ]:
import os

# 1. Hardcoded Credentials Anti-Pattern
class VulnerableClient:
    """Class showcasing dangerous hardcoded secrets."""
    # DANGEROUS: Committing credentials to codebase
    DB_PASSWORD = "SuperSecretDBPassword123!"
    API_KEY = "xyz-api-token-value"

    def connect(self):
        print(f"Connecting to database using key: {self.API_KEY}...")

# Test vulnerable client
client = VulnerableClient()
client.connect()



In [ ]:
# 2. Secure Config Loader using Environment Variables
class SecureConfigLoader:
    """Class showcasing dynamic environment configuration loading."""
    
    @classmethod
    def get_db_password(cls):
        # Fall back to None or a secure default if variable is missing
        # True secrets should be set in the host OS environment
        return os.environ.get("APP_DB_PASSWORD")

    @classmethod
    def get_api_key(cls):
        return os.environ.get("APP_API_KEY")

print("--- Testing Environment Load (Unset State) ---")
print(f"Secret API Key: {SecureConfigLoader.get_api_key()}")  # Expected: None (Secure default)



In [ ]:
# 3. Simulating .env parser config file load
# We simulate how libraries like python-dotenv parse key-value lines securely.
def load_mock_dotenv(dotenv_content):
    """Parses a simulated .env file content and injects keys into os.environ."""
    print("\n--- Parsing .env config ---")
    for line in dotenv_content.strip().split('\n'):
        # Ignore comments and empty lines
        if not line or line.startswith('#'):
            continue
            
        if '=' in line:
            key, val = line.split('=', 1)
            key = key.strip()
            val = val.strip().strip('"').strip("'")  # Clean quotes
            
            # Inject key into process environment
            os.environ[key] = val
            print(f"Loaded config key: {key}")

# Simulated .env file contents
dotenv_mock = """
# App Environment configurations
APP_DB_PASSWORD=MySecureEnvPassword999
APP_API_KEY=env-injected-token-abc
"""

# Load keys
load_mock_dotenv(dotenv_mock)

print("\n--- Testing Environment Load (Injected State) ---")
print(f"Secret API Key: {SecureConfigLoader.get_api_key()}")      # Expected: env-injected-token-abc
print(f"DB Password:   {SecureConfigLoader.get_db_password()}")  # Expected: MySecureEnvPassword999

# Clean up injected variables from process
for key in ["APP_DB_PASSWORD", "APP_API_KEY"]:
    if key in os.environ:
        del os.environ[key]
